# Parser Comparison

## PDF parser

### pdfpumber vs pyPDF

In [1]:
import os 
import sys
import re
from PyPDF2 import PdfReader
import pdfplumber 

In [2]:
# Set the root directory
os.chdir("..")                          # Change cwd for notebook
sys.path.append(os.path.abspath(".."))  # go to project root to look for python modules

In [28]:
def extract_resume_text_pdfplumber(file_path):
    """
    This function extract the content from pdf using and return text
    """
    text = ""
       
    try:
        with pdfplumber.open(file_path) as pdf:
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                else:
                    print(f"Warning: page {i+1} returned no text")

    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {file_path}")
    
    except ValueError:
        raise ValueError()                                       # allow validation error to pass through
    
    except Exception as e:
        raise RuntimeError(f"Error extracting text from PDF: {e}")
    
    # Check for scanned or empty PDF
    if len(text.strip())<100:
        print("PDF appears to be scanned or empty. Please upload a text-based PDF file")
    
    return text.strip()

In [ ]:
resume_text = extract_resume_text_pdfplumber(r"data\resume\harmandeep_singh_resume.pdf")
# print("pdfplumber output:\n",resume_text[:1000])
print("pdfplumber output:", resume_text[146:639])

#### PyPDF2

In [ ]:
def extract_resume_text_pyPDF2(file_path):
    text = ""
    try:
        reader = PdfReader(file_path)
         
        for page in reader.pages:
              page_text = page.extract_text()
              if page_text:
                   text += page_text + "\n"
        return text
    
    except Exception as e:
         raise RuntimeError(f"Error reading PDF: {e}")

In [24]:
resume_text_pyPDF2 = extract_resume_text_pyPDF2(r"data\resume\harmandeep_singh_resume.pdf")
# print("pyPDF2 output:", resume_text_pyPDF2[:1000])
print("pyPDF2 output:", resume_text_pyPDF2[390:901])

pyPDF2 output: 
EXPERIENCE  
Data Science & Machine Learning Bootcamp Trainee  Ironhack Gmbh | 12/2025 – 03/2025  
- Currently work on project -based data science and machine learning assignments . 
- Perform  data cleaning, exploratory data analysis (EDA), and feature engineering using Python  and Pandas . 
- Built and evaluated ML model (Linear/Logistic Regression, KNN, Decision Trees) . 
- Collabora te using Git/GitHub and follow agile -style workflows . 
- Present findings and model results to peers and instructors .


**Conclusion**

`pdfplumber` performs better than `PyPDF2` when extracting text from the PDF. It preserves the layout and spacing more accurately, resulting in cleaner and more structured text. Therefore, the PDF parser will be implemented using `pdfplumber`.


In [29]:
# Test empty file
empty_file_text = extract_resume_text_pdfplumber(r"data\resume\sample_resume.pdf")
print("pyPDF2 output:", empty_file_text[:1000])

PDF appears to be scanned or empty. Please upload a text-based PDF file
pyPDF2 output: 


## Docx Parser

In [47]:
from docx import Document

def extract_from_docx(file_path):
    """
    Extract text content from a DOCX file using python-docx.
    Extracts from both paragraphs and tables to capture
    all resume content including structured skill tables.
    
    Args:
        file_path (str): Path to the DOCX file
        
    Returns:
        str: Extracted and cleaned text content
        
    Raises:
        FileNotFoundError : If file does not exist
        ValueError        : If document is empty
        RuntimeError      : If extraction fails
    """
    try:
        doc = Document(file_path)

        lines = []

        # Paragraphs
        for para in doc.paragraphs:
            text = para.text.strip()
            if text:
                lines.append(text)

        # Tables
        for table in doc.tables:
            for row in table.rows:
                cells = []
                for cell in row.cells:
                    cell_text = cell.text.strip()
                    if cell_text:
                        cells.append(cell_text)
                if cells:
                    lines.append(" | ".join(cells))

        # Headers and Footers
        for section in doc.sections:

            for para in section.header.paragraphs:
                text = para.text.strip()
                if text:
                    lines.append(text)

            for para in section.footer.paragraphs:
                text = para.text.strip()
                if text:
                    lines.append(text)

        # Remove duplicates while preserving order
        seen = set()
        clean_lines = []
        for line in lines:
            if line not in seen:
                seen.add(line)
                clean_lines.append(line)

        text = "\n".join(clean_lines)

        if len(text.strip()) < 100:
            raise ValueError(
                "DOCX appears to be empty or contains insufficient content"
            )

        return text.strip()

    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {file_path}")

    except ValueError as e:
        raise ValueError(str(e))

    except Exception as e:
        raise RuntimeError(f"Error extracting text from DOCX: {e}")

In [50]:
docx_resume_text = extract_from_docx(r"data\resume\harmandeep.docx")
# print(docx_resume_text)
print("docx output:\n", docx_resume_text[:1000])

docx output:
 Plaussiger Str. 04318, Leipzig, Germany
EXPERIENCE
Data Science & Machine Learning Bootcamp (Project-Based)
Ironhack Gmbh | 12/2025 – 03/2026
-  Built end-to-end machine learning workflows including data ingestion, cleaning, feature engineering, model training, evaluation, and basic cloud deployment preparation.
- Performed exploratory data analysis to understand distributions, correlations, missing data patterns, and feature importance.
- Apply statistical techniques to identify trends and predictive patterns.
- Perform data cleaning, exploratory data analysis (EDA), and feature engineering using Python and Pandas.
- Built and evaluated ML model (Regression, Decision Trees, Random Forest, Boost ).
- Collaborate using Git/GitHub and follow agile-style workflows.
- Present analytical findings to technical and non-technical audience.
- Automate repetitive analysis workflows to improve reproducibility.
Operations Associate Analyst, Tier Mobility SE, Berlin, Germany | 10/2021

#### Docx to PDF Conversion

In [ ]:
from pathlib import Path
from docx2pdf import convert

def extract_from_docx_(file_path):

    path = Path(file_path)

    if path.suffix == ".docx" | path.suffix == ".doc":
        pdf_path = path.with_suffix(".pdf")
        convert(file_path, pdf_path)
        file_path = pdf_path

    if str(file_path).endswith(".pdf"):
        with pdfplumber.open(file_path) as pdf:
            return "\n".join(page.extract_text() for page in pdf.pages)

    if str(file_path).endswith(".txt"):
        return open(file_path).read()

    raise ValueError("Unsupported file format")

In [52]:
extract_from_docx_(r"data\resume\harmandeep.docx")

100%|██████████| 1/1 [00:03<00:00,  3.62s/it]


'Harmandeep Singh, M.Sc.\n(+49)015901768404 | Plaussiger Str. 04318, Leipzig, Germany\nharmandeep.singh95@outlook.com | LinkedIn, Tableau, | Portfolio Link\nEXPERIENCE\nData Science & Machine Learning Bootcamp (Project-Based)\nIronhack Gmbh | 12/2025 – 03/2026\n- Built end-to-end machine learning workflows including data ingestion, cleaning, feature engineering, model training,\nevaluation, and basic cloud deployment preparation.\n- Performed exploratory data analysis to understand distributions, correlations, missing data patterns, and feature\nimportance.\n- Apply statistical techniques to identify trends and predictive patterns.\n- Perform data cleaning, exploratory data analysis (EDA), and feature engineering using Python and Pandas.\n- Built and evaluated ML model (Regression, Decision Trees, Random Forest, Boost ).\n- Collaborate using Git/GitHub and follow agile-style workflows.\n- Present analytical findings to technical and non-technical audience.\n- Automate repetitive analys

In [55]:
resume_text = extract_resume_text_pdfplumber(r"data\resume\harmandeep.pdf")
print(resume_text[:1000])

Harmandeep Singh, M.Sc.
(+49)015901768404 | Plaussiger Str. 04318, Leipzig, Germany
harmandeep.singh95@outlook.com | LinkedIn, Tableau, | Portfolio Link
EXPERIENCE
Data Science & Machine Learning Bootcamp (Project-Based)
Ironhack Gmbh | 12/2025 – 03/2026
- Built end-to-end machine learning workflows including data ingestion, cleaning, feature engineering, model training,
evaluation, and basic cloud deployment preparation.
- Performed exploratory data analysis to understand distributions, correlations, missing data patterns, and feature
importance.
- Apply statistical techniques to identify trends and predictive patterns.
- Perform data cleaning, exploratory data analysis (EDA), and feature engineering using Python and Pandas.
- Built and evaluated ML model (Regression, Decision Trees, Random Forest, Boost ).
- Collaborate using Git/GitHub and follow agile-style workflows.
- Present analytical findings to technical and non-technical audience.
- Automate repetitive analysis workflows to 

**Docx Parser Conclusion**

Direct parsing of ``.docx`` files using python-docx can miss important information. Many resumes store contact details and other elements inside headers, tables, text boxes, or layout objects that are not always reliably extracted by the DOCX parser.

A more reliable approach is to convert the DOCX file into a PDF and then extract the text using pdfplumber. PDF extraction typically captures the visible layout more consistently, which improves the completeness of the extracted content.

To implement this approach, the following libraries are required:

- ``pathlib`` for robust file path handling

- ``docx2pdf`` to convert DOCX files to PDF